# Contrastive RL (Acme-free port) — Colab runner

Thin notebook: all logic lives in the `crl/` package. This notebook only wires
up Colab (deps, Drive, GPU) and calls into it.

**No dataset needed** — this is *online* contrastive RL: the agent generates its
own data by interacting with the simulator during training.

**Expected wall-clock on a T4:** `point_*` ~1–3 min; `fetch_reach` roughly
**20–40 min** for ~300k steps (watch the printed `sps=` — time ≈ steps / sps).
The loop is collection-bound (MuJoCo on CPU + per-step dispatch), so raising
`num_sgd_steps_per_step` uses the GPU better. See the last markdown cell.

In [ ]:
# 1. Install dependencies (~2 min).
!pip -q install "jax[cuda12]" dm-haiku optax gymnasium gymnasium-robotics mujoco imageio
import os
os.environ["MUJOCO_GL"] = "egl"  # headless MuJoCo rendering backend on Colab.
import jax; print("JAX devices:", jax.devices())

In [ ]:
# 2. Get the code (your fork).
import os
if not os.path.exists("contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd contrastive_rl

In [ ]:
# 3. Mount Google Drive and pick a folder for checkpoints + results.
from google.colab import drive
drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/contrastive_rl_runs/fetch_reach_nce"
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoints/metrics will be written to:", CKPT_DIR)

In [ ]:
# 4. (Optional, ~1 min) Sanity-check the whole pipeline on the pure-numpy
#    point env before spending GPU time on Fetch.
!python -m crl.selftest
!python -m crl.checks | tail -5

In [ ]:
# 5. Train with the BINARY NCE / pure contrastive objective (use_td=False,
#    twin_q=False) -- this is the objective the causal method builds on.
#    Checkpoints + metrics.json save to Drive on every eval.
from crl.config import Config
from crl.train import train

config = Config(
    env_name="fetch_reach",
    use_td=False, twin_q=False,   # binary NCE (contrastive_nce)
    entropy_coefficient=0.0,      # state-based task (matches the original)
    max_number_of_steps=300_000,
    min_replay_size=10_000,
    random_steps=10_000,
    num_sgd_steps_per_step=4,     # batch more grad steps per env step (uses GPU)
    eval_every_steps=10_000,
    eval_episodes=20,
    ckpt_dir=CKPT_DIR,            # periodic saves to Drive
    resume=False,                 # fresh run (NCE critic shape != C-learning's)
)
state = train(config)

In [ ]:
# 6. Watch the agent: render a rollout of the BEST checkpoint to a GIF.
from crl.visualize import rollout_gif
gif = rollout_gif(
    "fetch_reach",
    ckpt=os.path.join(CKPT_DIR, "best.pkl"),  # or None for a random policy
    out=os.path.join(CKPT_DIR, "rollout.gif"),
    episodes=3,
)
from IPython.display import Image
Image(gif)

In [ ]:
# 7. Plot success-rate over training from the saved metrics.
import json, matplotlib.pyplot as plt
with open(os.path.join(CKPT_DIR, "metrics.json")) as f:
    hist = json.load(f)
plt.plot([h["step"] for h in hist], [h["success"] for h in hist], "-o")
plt.xlabel("env steps"); plt.ylabel("success rate"); plt.grid(True); plt.show()

## Notes
- **Time on a T4:** dominated by data collection + per-step dispatch, not the
  gradient math. `point_*` finishes in a couple minutes; `fetch_reach` typically
  reaches good success in **~20–40 min** (~300k steps). `fetch_push` is harder —
  budget more steps. The real rate is the printed `sps` — estimate time as
  `max_number_of_steps / sps` seconds.
- **If you disconnect:** re-run cells 1–5 with the same `CKPT_DIR` and
  `resume=True` — it reloads `latest.pkl` and continues.
- **Checkpoints written:** `latest.pkl` (resume), `best.pkl` (highest eval
  success), `metrics.json` (full success history).